In [51]:
import pandas as pd
import glob
import os
import io


os.chdir('/Users/ijaejun/Documents/sophomore_high/crime_catchers')
print("현재 위치:", os.getcwd())
print("파일 목록:", glob.glob('data/raw/경찰청/*.csv'))
print(os.getcwd())
print(glob.glob('data/raw/e지방지표/*.csv'))

현재 위치: /Users/ijaejun/Documents/sophomore_high/crime_catchers
파일 목록: ['data/raw/경찰청/경찰청_범죄_발생_2016.csv', 'data/raw/경찰청/경찰청_범죄_발생_2017.csv', 'data/raw/경찰청/경찰청_범죄_발생_2015.csv', 'data/raw/경찰청/경찰청_범죄_발생_2014.csv', 'data/raw/경찰청/경찰청_범죄_발생_2023.csv', 'data/raw/경찰청/경찰청_범죄_발생_2022.csv', 'data/raw/경찰청/경찰청_범죄_발생_2020.csv', 'data/raw/경찰청/경찰청_범죄_발생_2021.csv', 'data/raw/경찰청/경찰청_범죄_발생_2019.csv', 'data/raw/경찰청/경찰청_범죄_발생_2024.csv', 'data/raw/경찰청/경찰청_범죄_발생_2018.csv']
/Users/ijaejun/Documents/sophomore_high/crime_catchers
['data/raw/e지방지표/대구_e지방지표.csv', 'data/raw/e지방지표/부산_e지방지표.csv', 'data/raw/e지방지표/인천_e지방지표.csv', 'data/raw/e지방지표/대전_e지방지표.csv', 'data/raw/e지방지표/광주_e지방지표.csv', 'data/raw/e지방지표/울산_e지방지표.csv']


In [55]:
# ============================================================
# Step 1. e지방지표 - 독립변수 합치기
# ============================================================
 
# 필요한 지표 목록 -- 독립변수
INDICATORS = {
    '실업률(시도) (%)':           '실업률',
    '음주율 (%)':                 '음주율',
    '소비자물가 등락률 (%)':       '물가상승률',
    '주민등록인구 (명)':           '인구수',
    '기온 (℃)':                  '평균기온',
    '경찰공무원 1인당 담당주민수 (명)': '경찰1인당주민수',
    '국민기초생활보장 수급자수 (명)': '기초수급자수',
    '조이혼율':                   '조이혼율',
    '1인당 GRDP (천원)':          '지역소득',
    '등록외국인 현황 (명)':        '외국인수',
}

YEARS = [str(y) for y in range(2014, 2025)]  # 2014~2024
e_files = glob.glob('data/raw/e지방지표/*.csv')
e_all = []

for file in e_files:
    city = os.path.basename(file).replace('_e지방지표.csv', '')
    df = pd.read_csv(file, encoding='utf-8-sig')

 
    # 필요한 지표만 필터링
    df_filtered = df[df['지표별'].isin(INDICATORS.keys())].copy()
 
    # 연도 컬럼만 추출
    year_cols = [y for y in YEARS if y in df_filtered.columns]
    df_filtered = df_filtered[['지표별'] + year_cols]
 
    # 세로로 변환 (wide → long)
    df_long = df_filtered.melt(
        id_vars='지표별',
        value_vars=year_cols,
        var_name='연도',
        value_name='값'
    )
 
    df_long['지역'] = city
    df_long['연도'] = df_long['연도'].astype(int)
    df_long['지표별'] = df_long['지표별'].map(INDICATORS)
    e_all.append(df_long)

    # 위 코드 그대로 + 아래 추가!

df_e = pd.concat(e_all, ignore_index=True)

df_e_pivot = df_e.pivot_table(
    index=['지역', '연도'],
    columns='지표별',
    values='값',
    aggfunc='first'
).reset_index()

df_e_pivot.columns.name = None

print("✅ e지방지표 완료!")
print(df_e_pivot.shape)
print(df_e_pivot.head())

✅ e지방지표 완료!
(66, 12)
   지역    연도 경찰1인당주민수 기초수급자수 물가상승률  실업률   외국인수   음주율      인구수 조이혼율   지역소득  평균기온
0  광주  2014    476.7  59598   1.6  2.8  17064  60.6  1475884  2.1  23448  14.3
1  광주  2015    458.6  71683   0.3  2.9  18455  61.9  1472199  1.9  24731  14.6
2  광주  2016    454.6  69420   0.9  3.1  19920  58.6  1469214  1.9  26248  15.0
3  광주  2017    447.5  65712   2.1  2.9  21279  61.6  1463770  1.8  27449  14.6
4  광주  2018    439.6  72757   1.2  3.8  22815  60.3  1459336  2.0  28594  14.6


In [56]:
CRIMES_5 = ['살인기수', '강도', '강간', '절도범죄', '폭행']
CITIES   = ['부산', '대구', '인천', '광주', '대전', '울산']
 
crime_files = glob.glob('data/raw/경찰청/*.csv')
crime_all = []

for file in sorted(crime_files):
    with open(file, 'rb') as f:
        raw = f.read()
    text = raw.decode('cp949', errors='replace')
    df = pd.read_csv(io.StringIO(text))
    
    # 컬럼명 정확히 출력
    print(os.path.basename(file))
    print(repr(df.columns[0]))  # 첫번째 컬럼 정확한 이름
    print(repr(df.columns[1]))  # 두번째 컬럼 정확한 이름
    print('---')

경찰청_범죄_발생_2017.csv
'범죄대분류'
'범죄중분류'
---
경찰청_범죄_발생_2024.csv
'범죄대분류'
'범죄중분류'
---
경찰청_범죄_발생_2014.csv
'범죄대분류'
'범죄중분류'
---
경찰청_범죄_발생_2015.csv
'범죄대분류'
'범죄중분류'
---
경찰청_범죄_발생_2016.csv
'범죄대분류'
'범죄중분류'
---
경찰청_범죄_발생_2018.csv
'범죄대분류'
'범죄중분류'
---
경찰청_범죄_발생_2019.csv
'범죄대분류'
'범죄중분류'
---
경찰청_범죄_발생_2020.csv
'범죄대분류'
'범죄중분류'
---
경찰청_범죄_발생_2021.csv
'범죄대분류'
'범죄중분류'
---
경찰청_범죄_발생_2022.csv
'범죄대분류'
'범죄중분류'
---
경찰청_범죄_발생_2023.csv
'범죄대분류'
'범죄중분류'
---


In [62]:
CRIMES_5 = ['살인기수', '강도', '강간', '절도', '폭행']
CITIES = ['부산', '대구', '인천', '광주', '대전', '울산']

crime_files = glob.glob('data/raw/경찰청/*.csv')
crime_all = []

for file in sorted(crime_files):
    year = int(''.join(filter(str.isdigit, os.path.basename(file)))[:4])

    df = None
    for enc in ['cp949', 'utf-8-sig', 'utf-8']:
        try:
            temp = pd.read_csv(file, encoding=enc)
            if '범죄중분류' in temp.columns:
                df = temp
                break
        except:
            continue

    if df is None:
        print(f"❌ 실패: {file}")
        continue

    for city in CITIES:
        city_cols = [c for c in df.columns if c.startswith(city)]
        row = {'지역': city, '연도': year}

        # 범죄별 개별 집계
        for crime in CRIMES_5:
            df_crime_row = df[df['범죄중분류'] == crime]
            if not df_crime_row.empty:
                row[crime] = int(df_crime_row[city_cols].sum().sum())
            else:
                row[crime] = 0

        crime_all.append(row)

df_crime = pd.DataFrame(crime_all).sort_values(['연도', '지역']).reset_index(drop=True)
print("✅ 완료!")
print(df_crime.head(6))

✅ 완료!
   지역    연도  살인기수   강도   강간     절도    폭행
0  광주  2014    12   53  220  10361  5718
1  대구  2014    17   63  219  14609  5280
2  대전  2014    12   82  168  11453  2457
3  부산  2014    21  131  399  21898  7821
4  울산  2014    13   32  126   5777  3654
5  인천  2014    16  111  308   9737  8300


In [59]:
# ============================================================
# Step 3. 두 데이터 합치기 (merge)
# ============================================================
 
df_merged = pd.merge(df_e_pivot, df_crime, on=['지역', '연도'], how='inner')

# 숫자 변환
num_cols = ['외국인수', '인구수', '실업률', '음주율', '물가상승률',
            '평균기온', '경찰1인당주민수', '기초수급자수', '조이혼율', '지역소득']
for col in num_cols:
    df_merged[col] = pd.to_numeric(df_merged[col], errors='coerce')

# 외국인 비율
df_merged['외국인비율(%)'] = (
    df_merged['외국인수'] / df_merged['인구수'] * 100
).round(4)

print("✅ 최종 완료!")
print(df_merged.shape)
print(df_merged.head())

# 저장
import os
os.makedirs('data/processed', exist_ok=True)
df_merged.to_csv('data/processed/merged_data.csv', index=False, encoding='utf-8-sig')
print("💾 저장 완료!")
 
# 컬럼 순서 정리
cols_order = [
    '지역', '연도',
    '5대범죄합계',        # 종속변수
    '실업률', '음주율', '물가상승률', '인구수',
    '평균기온', '경찰1인당주민수', '기초수급자수',
    '조이혼율', '지역소득', '외국인수', '외국인비율(%)'
]
df_merged = df_merged[cols_order]
 
print("\n✅ 최종 병합 완료!")
print(df_merged.shape)
print(df_merged)
 

✅ 최종 완료!
(66, 14)
   지역    연도  경찰1인당주민수  기초수급자수  물가상승률  실업률   외국인수   음주율      인구수  조이혼율   지역소득  \
0  광주  2014     476.7   59598    1.6  2.8  17064  60.6  1475884   2.1  23448   
1  광주  2015     458.6   71683    0.3  2.9  18455  61.9  1472199   1.9  24731   
2  광주  2016     454.6   69420    0.9  3.1  19920  58.6  1469214   1.9  26248   
3  광주  2017     447.5   65712    2.1  2.9  21279  61.6  1463770   1.8  27449   
4  광주  2018     439.6   72757    1.2  3.8  22815  60.3  1459336   2.0  28594   

   평균기온  5대범죄합계  외국인비율(%)  
0  14.3    6003    1.1562  
1  14.6    5657    1.2536  
2  15.0    5023    1.3558  
3  14.6    5095    1.4537  
4  14.6   10070    1.5634  
💾 저장 완료!

✅ 최종 병합 완료!
(66, 14)
    지역    연도  5대범죄합계  실업률   음주율  물가상승률      인구수  평균기온  경찰1인당주민수  기초수급자수  조이혼율  \
0   광주  2014    6003  2.8  60.6    1.6  1475884  14.3     476.7   59598   2.1   
1   광주  2015    5657  2.9  61.9    0.3  1472199  14.6     458.6   71683   1.9   
2   광주  2016    5023  3.1  58.6    0.9  1469214  15.0     4

In [60]:
# ============================================================
# Step 4. 저장
# ============================================================
 
os.makedirs('data/processed', exist_ok=True)
df_merged.to_csv('data/processed/merged_data.csv',
                 index=False, encoding='utf-8-sig')
print("\n💾 data/processed/merged_data.csv 저장 완료!")


💾 data/processed/merged_data.csv 저장 완료!
